In [ ]:
import pandas as pd
import numpy as np
from plot_utils import time_gain_plot

color_palette = ['#C0D6CA',
                 '#78ACA8',
                 '#2D6B8F',
                 '#235796',
                 '#E7C4C0',
                 '#E3A39A',
                 '#CA6F6A',
                 '#7B3841',
                 '#D5BC67',
                 '#20425B',
                 '#E77A5B',
                 '#9C9DB2',
                ]


In [ ]:
def find_closest_timestamp(target_timestamp, possible_timestamps):
    abs_diff = (possible_timestamps - target_timestamp).abs()
    return possible_timestamps.iloc[abs_diff.argmin()]

def get_time_gain(data, id_):
    critical_val = 180 #hyperglycemia
    ct_baseline = []
    ct_pk = []
    ct_all = []
    for trial in range(8):
        forecast_baseline = pd.read_csv(f'{baseline_results_path}/ohiot1dm_exog_6/baseline_models/trial_{trial}/forecasts.csv')
        forecast_baseline.ds = pd.to_datetime(forecast_baseline.ds, format='%Y-%m-%d %H:%M:%S')
        forecast_baseline.cutoff = pd.to_datetime(forecast_baseline.cutoff, format='%Y-%m-%d %H:%M:%S')
        forecast_baseline = forecast_baseline.groupby(['unique_id', 'cutoff']).tail(1) 
        forecast_baseline = forecast_baseline[forecast_baseline.unique_id==id_].copy()
    
        forecast_pk = pd.read_csv(f'{pk_results_path}/ohiot1dm_exog_6/pk_models/trial_{trial}/forecasts.csv')
        forecast_pk.ds = pd.to_datetime(forecast_pk.ds, format='%Y-%m-%d %H:%M:%S')
        forecast_pk.cutoff = pd.to_datetime(forecast_pk.cutoff, format='%Y-%m-%d %H:%M:%S')
        forecast_pk = forecast_pk.groupby(['unique_id', 'cutoff']).tail(1) 
        forecast_pk = forecast_pk[forecast_pk.unique_id==id_].copy()

        # Restrict evaluation to the test set
        d2 = data[(data.unique_id==id_)&(data.ds>forecast_pk.ds.min())].copy()

        # Identify start of hyperglycemic events where glucose cross threshold of 180.
        crossings = d2[(d2['y'] >= critical_val) & (d2['y_shifted'] < critical_val)] 
        crossings = crossings[crossings.y<critical_val+5]

        # Find times where models predict hyperglycemic events (>=180)
        baseline_pbd = forecast_baseline[forecast_baseline['AutoNHITS'] >= critical_val]
        pk_pbd = forecast_pk[forecast_pk['AutoNHITS_TREAT'] >= critical_val]

        # Find the model prediction timestamps that are closest to the actual event time
        for crossing_time in crossings.ds:
            ct_baseline.append(find_closest_timestamp(crossing_time, baseline_pbd['ds']))
            ct_pk.append(find_closest_timestamp(crossing_time, pk_pbd['ds']))
            ct_all.append(crossing_time)

    # Get time difference from glucose threshold crossing reference time
    diff_baseline = [(i-j).astype('timedelta64[s]') for i, j in zip(pd.Series(ct_baseline).values, pd.Series(ct_all).values)]
    diff_pk = [(i-j).astype('timedelta64[s]') for i, j in zip(pd.Series(ct_pk).values, pd.Series(ct_all).values)]

    # Convert time gain to minutes
    mean_baseline = np.mean(diff_baseline) / np.timedelta64(1, 'm')
    mean_treat = np.mean(diff_pk) / np.timedelta64(1, 'm')
    
    return mean_baseline, mean_treat
    

In [ ]:
data = pd.read_csv('../datasets/ohiot1dm_exog_9_day_test.csv')
data.ds = pd.to_datetime(data.ds, format='%Y-%m-%d %H:%M:%S')
data['y_shifted'] = data['y'].shift(1)
ids = ['#540', '#544', '#552', '#559', '#563', '#567', '#570', '#575', '#584', '#588', '#591', '#596']

baseline_results_path = '../results/multivariate_models'
pk_results_path = '../results/multivariate_models'

baseline_model = []
pk_model = []
for id_ in ids:
    print(id_)
    mean_baseline, mean_pk = get_time_gain(data, id_)
    baseline_model.append(mean_baseline)
    pk_model.append(mean_pk)

values = np.array(baseline_model) - np.array(pk_model)

# Generate plot
time_gain_plot(ids, values, save_dir=None)